In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pprint import pprint
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path(".").resolve() / ".env")

sys.path.insert(0, str(Path("src").resolve()))

from compute_tracker import find_compute_series, get_markets_by_series, safe_get
from config import (
    KALSHI_CANDLE_PERIOD_DAILY,
    KALSHI_CANDLESTICKS,
    KALSHI_MARKETS,
    KALSHI_PAGE_LIMIT,
    KALSHI_REQUEST_TIMEOUT_SEC,
    KEYWORDS,
    kalshi_url,
)

In [3]:
# Pull Kalshi series whose titles match GPU/compute keywords
compute_series = find_compute_series()
compute_series

GET /series -> HTTP 200
Total series collected: 13631
Categories seen: ['Climate and Weather', 'Commodities', 'Companies', 'Crypto', 'Economics', 'Education', 'Elections', 'Entertainment', 'Exotics', 'Financials', 'Health', 'Mentions', 'Politics', 'Science and Technology', 'Social', 'Sports', 'Transportation', 'World']
Series matching keywords ['gpu', 'compute', 'b200', 'h200', 'a100', 'h100', 'nvidia']: 32
  KXNVIDIARASGONQ: NVIDIA Rasgon Q
  KXB200W: B200 Weekly Price
  KXH200MAX: H200 Yearly Directional Price
  KXNVDAA: Nvidia KPI
  KXB200WS: B200 Weekly
  KXH100W: H100 Weekly Price
  KXB200TEST: B200 Weekly
  KXH200Q: H200 Quarterly Directional Price
  KXA100W: A100 Weekly Price
  KXH200MS: H200 Monthly
  KXA100MS: A100 Monthly
  KXB200AVGTEST: B200 Monthly Average
  KXTESTB200: Weekly B200 Compute Price
  KXB200Q: B200 Quarterly Directional Price
  KXA100MAX: A100 Yearly Directional Price
  KXA100MON: A100 Monthly Price
  KXH100MON: H100 Monthly Price
  KXH200CHINA: Nvidia H200 ch

[{'additional_prohibitions': ['Persons who are employed by any of the Source Agencies are not permitted to trade on the Contract.',
   'Persons who hold any material, non-public information on the Underlying are not permitted to trade on the Contract.'],
  'category': 'Social',
  'contract_terms_url': 'https://assets.kalshi.com/contract_terms/ASKQUESTION.pdf',
  'contract_url': 'https://assets.kalshi.com/regulatory/product-certifications/ASKQUESTION.pdf',
  'exchange_index': 0,
  'fee_multiplier': 1,
  'fee_type': 'quadratic',
  'frequency': 'one_off',
  'last_updated_ts': '2026-02-26T08:50:31.901806Z',
  'settlement_sources': [{'name': 'ABC', 'url': 'https://kalshi.com/'},
   {'name': 'Axios', 'url': 'https://www.axios.com/'},
   {'name': 'Bloomberg News', 'url': 'https://www.bloomberg.com/news'},
   {'name': 'CBS', 'url': 'https://www.cbs.com'},
   {'name': 'CNBC', 'url': 'https://www.cnbc.com'},
   {'name': 'CNN', 'url': 'https://www.cnn.com'},
   {'name': 'Financial Times', 'url': 

In [4]:
# Compact view: ticker + title + category
markets = [(s.get("ticker"), s.get("title"), s.get("category")) for s in compute_series]
markets

[('KXNVIDIARASGONQ', 'NVIDIA Rasgon Q', 'Social'),
 ('KXB200W', 'B200 Weekly Price', 'Science and Technology'),
 ('KXH200MAX', 'H200 Yearly Directional Price', 'Science and Technology'),
 ('KXNVDAA', 'Nvidia KPI', 'Financials'),
 ('KXB200WS', 'B200 Weekly', 'Financials'),
 ('KXH100W', 'H100 Weekly Price', 'Science and Technology'),
 ('KXB200TEST', 'B200 Weekly', 'Science and Technology'),
 ('KXH200Q', 'H200 Quarterly Directional Price', 'Science and Technology'),
 ('KXA100W', 'A100 Weekly Price', 'Science and Technology'),
 ('KXH200MS', 'H200 Monthly', 'Financials'),
 ('KXA100MS', 'A100 Monthly', 'Financials'),
 ('KXB200AVGTEST', 'B200 Monthly Average', 'Financials'),
 ('KXTESTB200', 'Weekly B200 Compute Price', 'Financials'),
 ('KXB200Q', 'B200 Quarterly Directional Price', 'Science and Technology'),
 ('KXA100MAX', 'A100 Yearly Directional Price', 'Science and Technology'),
 ('KXA100MON', 'A100 Monthly Price', 'Science and Technology'),
 ('KXH100MON', 'H100 Monthly Price', 'Science an

In [5]:
print(compute_series[0])
print(markets[0])

{'additional_prohibitions': ['Persons who are employed by any of the Source Agencies are not permitted to trade on the Contract.', 'Persons who hold any material, non-public information on the Underlying are not permitted to trade on the Contract.'], 'category': 'Social', 'contract_terms_url': 'https://assets.kalshi.com/contract_terms/ASKQUESTION.pdf', 'contract_url': 'https://assets.kalshi.com/regulatory/product-certifications/ASKQUESTION.pdf', 'exchange_index': 0, 'fee_multiplier': 1, 'fee_type': 'quadratic', 'frequency': 'one_off', 'last_updated_ts': '2026-02-26T08:50:31.901806Z', 'settlement_sources': [{'name': 'ABC', 'url': 'https://kalshi.com/'}, {'name': 'Axios', 'url': 'https://www.axios.com/'}, {'name': 'Bloomberg News', 'url': 'https://www.bloomberg.com/news'}, {'name': 'CBS', 'url': 'https://www.cbs.com'}, {'name': 'CNBC', 'url': 'https://www.cnbc.com'}, {'name': 'CNN', 'url': 'https://www.cnn.com'}, {'name': 'Financial Times', 'url': 'https://www.ft.com'}, {'name': 'Fox Bus

In [6]:
get_markets_by_series(markets[0][0])

GET /markets -> HTTP 200


[]

In [7]:
from compute_tracker import SESSION

series_ticker = markets[0][0]  # e.g. KXB200Q

# Probe status values: "all" is what used to break; omit status for any-status
STATUS_CANDIDATES = [None, "all", "unopened", "open", "paused", "closed", "settled"]

def fetch_markets_raw(series_ticker, status=None):
    params = {"series_ticker": series_ticker, "limit": 5}
    if status is not None:
        params["status"] = status
    resp = SESSION.get(
        kalshi_url(KALSHI_MARKETS),
        params=params,
        timeout=KALSHI_REQUEST_TIMEOUT_SEC,
    )
    try:
        body = resp.json()
    except ValueError:
        body = {"_raw": resp.text[:500]}
    return resp.status_code, params, body

results = {}
for status in STATUS_CANDIDATES:
    code, params, body = fetch_markets_raw(series_ticker, status)
    n_markets = len(body.get("markets", [])) if isinstance(body, dict) else 0
    results[str(status)] = {
        "http": code,
        "params": params,
        "n_markets": n_markets,
        "error": body.get("error") if isinstance(body, dict) else None,
        "sample_statuses": [
            m.get("status") for m in body.get("markets", [])[:3]
        ] if isinstance(body, dict) and body.get("markets") else None,
    }

results


{'None': {'http': 200,
  'params': {'series_ticker': 'KXNVIDIARASGONQ', 'limit': 5},
  'n_markets': 0,
  'error': None,
  'sample_statuses': None},
 'all': {'http': 400,
  'params': {'series_ticker': 'KXNVIDIARASGONQ', 'limit': 5, 'status': 'all'},
  'n_markets': 0,
  'error': {'code': 'bad_request',
   'message': 'bad request',
   'details': 'invalid status filter'},
  'sample_statuses': None},
 'unopened': {'http': 200,
  'params': {'series_ticker': 'KXNVIDIARASGONQ',
   'limit': 5,
   'status': 'unopened'},
  'n_markets': 0,
  'error': None,
  'sample_statuses': None},
 'open': {'http': 200,
  'params': {'series_ticker': 'KXNVIDIARASGONQ', 'limit': 5, 'status': 'open'},
  'n_markets': 0,
  'error': None,
  'sample_statuses': None},
 'paused': {'http': 200,
  'params': {'series_ticker': 'KXNVIDIARASGONQ',
   'limit': 5,
   'status': 'paused'},
  'n_markets': 0,
  'error': None,
  'sample_statuses': None},
 'closed': {'http': 200,
  'params': {'series_ticker': 'KXNVIDIARASGONQ',
   'l

In [8]:
# Full JSON for the bad filter vs omit-status (any status)
for label, status in [("BAD: status=all", "all"), ("OK: no status", None), ("OK: status=open", "open")]:
    code, params, body = fetch_markets_raw(series_ticker, status)
    print(f"\n=== {label} → HTTP {code} ===")
    print("params:", params)
    pprint(body if code != 200 else {
        "cursor": body.get("cursor"),
        "n_markets": len(body.get("markets", [])),
        "markets": [
            {
                "ticker": m.get("ticker"),
                "title": m.get("title"),
                "status": m.get("status"),
                "yes_bid_dollars": m.get("yes_bid_dollars"),
                "last_price_dollars": m.get("last_price_dollars"),
                "yes_ask_dollars": m.get("yes_ask_dollars"),
            }
            for m in body.get("markets", [])[:3]
        ],
    })



=== BAD: status=all → HTTP 400 ===
params: {'series_ticker': 'KXNVIDIARASGONQ', 'limit': 5, 'status': 'all'}
{'error': {'code': 'bad_request',
           'details': 'invalid status filter',
           'message': 'bad request'}}

=== OK: no status → HTTP 200 ===
params: {'series_ticker': 'KXNVIDIARASGONQ', 'limit': 5}
{'cursor': '', 'markets': [], 'n_markets': 0}

=== OK: status=open → HTTP 200 ===
params: {'series_ticker': 'KXNVIDIARASGONQ', 'limit': 5, 'status': 'open'}
{'cursor': '', 'markets': [], 'n_markets': 0}


In [9]:
import time
from datetime import datetime, timezone

import pandas as pd


def get_all_markets_for_series(series_ticker, max_pages=20):
    """Paginate /markets for one series (no status filter)."""
    markets, cursor = [], None
    for _ in range(max_pages):
        params = {"series_ticker": series_ticker, "limit": KALSHI_PAGE_LIMIT}
        if cursor:
            params["cursor"] = cursor
        data = safe_get(KALSHI_MARKETS, params)
        if not data:
            break
        page = data.get("markets", [])
        markets.extend(page)
        cursor = data.get("cursor")
        if not cursor or not page:
            break
    return markets


def market_row(series, market):
    return {
        "series_ticker": series.get("ticker"),
        "series_title": series.get("title"),
        "market_ticker": market.get("ticker"),
        "title": market.get("title"),
        "status": market.get("status"),
        "yes_bid": market.get("yes_bid_dollars"),
        "yes_ask": market.get("yes_ask_dollars"),
        "last_price": market.get("last_price_dollars"),
        "previous_price": market.get("previous_price_dollars"),
        "volume": market.get("volume_fp"),
        "volume_24h": market.get("volume_24h_fp"),
        "open_interest": market.get("open_interest_fp"),
        "liquidity": market.get("liquidity_dollars"),
        "floor_strike": market.get("floor_strike"),
        "open_time": market.get("open_time"),
        "close_time": market.get("close_time"),
        "created_time": market.get("created_time"),
        "updated_time": market.get("updated_time"),
        "event_ticker": market.get("event_ticker"),
        "rules_primary": market.get("rules_primary"),
    }


In [10]:
# Pull every market under each compute series
all_markets = []
for series in compute_series:
    mkts = get_all_markets_for_series(series["ticker"])
    print(f"{series['ticker']}: {len(mkts)} markets")
    all_markets.extend(market_row(series, m) for m in mkts)

markets_df = pd.DataFrame(all_markets)
print(f"\nTotal markets: {len(markets_df)}")
print(markets_df["status"].value_counts().to_string())
markets_df.head(10)


GET /markets -> HTTP 200
KXNVIDIARASGONQ: 0 markets
GET /markets -> HTTP 200
KXB200W: 11 markets
GET /markets -> HTTP 200
KXH200MAX: 6 markets
GET /markets -> HTTP 200
KXNVDAA: 6 markets
GET /markets -> HTTP 200
GET /markets -> HTTP 200
KXB200WS: 218 markets
GET /markets -> HTTP 200
KXH100W: 11 markets
GET /markets -> HTTP 200
KXB200TEST: 0 markets
GET /markets -> HTTP 200
KXH200Q: 20 markets
GET /markets -> HTTP 200
KXA100W: 11 markets
GET /markets -> HTTP 200
KXH200MS: 165 markets
GET /markets -> HTTP 200
KXA100MS: 101 markets
GET /markets -> HTTP 200
KXB200AVGTEST: 0 markets
GET /markets -> HTTP 200
KXTESTB200: 0 markets
GET /markets -> HTTP 200
KXB200Q: 5 markets
GET /markets -> HTTP 200
KXA100MAX: 9 markets
GET /markets -> HTTP 200
KXA100MON: 80 markets
GET /markets -> HTTP 200
KXH100MON: 40 markets
GET /markets -> HTTP 200
KXH200CHINA: 1 markets
GET /markets -> HTTP 200
KXB200MS: 141 markets
GET /markets -> HTTP 200
KXH100MS: 130 markets
GET /markets -> HTTP 200
KXFLOP: 0 markets

,series_ticker,series_title,market_ticker,title,status,yes_bid,yes_ask,last_price,previous_price,volume,volume_24h,open_interest,liquidity,floor_strike,open_time,close_time,created_time,updated_time,event_ticker,rules_primary
0,KXB200W,B200 Weekly Price,KXB200W-26SEP04-6.01,B200 price up in next week?,active,0.4900,0.5900,0.0000,0.0000,0.00,0.00,0.00,0.0000,6.01,2026-08-28T21:00:00Z,2026-09-04T21:00:00Z,2026-08-28T20:30:29.754601Z,2026-08-28T21:00:00.902454Z,KXB200W-26SEP04,If the value of B200 compute per hour is above...
1,KXB200W,B200 Weekly Price,KXB200W-26AUG28-6.95,B200 price up in next week?,closed,0.0000,1.0000,0.0600,0.0600,1057.22,0.00,1001.00,0.0000,6.95,2026-08-21T21:00:00Z,2026-08-28T21:00:00Z,2026-08-21T20:30:28.937259Z,2026-08-28T21:00:00.716753Z,KXB200W-26AUG28,If the value of B200 compute per hour is above...
2,KXB200W,B200 Weekly Price,KXB200W-26AUG21-6,B200 price up in next week?,finalized,0.0000,1.0000,0.9900,0.9900,1892.42,0.00,1665.00,0.0000,6.00,2026-08-14T21:00:00Z,2026-08-21T21:00:00Z,2026-08-14T20:30:27.559623Z,2026-08-21T21:35:27.282788Z,KXB200W-26AUG21,If the value of B200 compute per hour is above...
3,KXB200W,B200 Weekly Price,KXB200W-26AUG14-5.76,B200 price up in next week?,finalized,0.0000,1.0000,0.9900,0.9900,4117.77,0.00,1665.29,0.0000,5.76,2026-08-07T21:00:00Z,2026-08-14T21:00:00Z,2026-08-07T20:30:28.625901Z,2026-08-14T21:35:32.43175Z,KXB200W-26AUG14,If the value of B200 compute per hour is above...
4,KXB200W,B200 Weekly Price,KXB200W-26AUG07-6.41,B200 price up in next week?,finalized,0.0000,1.0000,0.0400,0.0400,1138.64,0.00,1080.93,0.0000,6.41,2026-07-31T21:00:00Z,2026-08-07T21:00:00Z,2026-07-31T20:31:52.444367Z,2026-08-07T21:35:35.500898Z,KXB200W-26AUG07,If the value of B200 compute per hour is above...
5,KXB200W,B200 Weekly Price,KXB200W-26JUL31-6.78,B200 price up in next week?,finalized,0.0000,1.0000,0.0400,0.0400,1109.03,0.00,813.92,0.0000,6.78,2026-07-24T21:00:00Z,2026-07-31T21:00:00Z,2026-07-24T20:30:26.187152Z,2026-07-31T21:35:40.918471Z,KXB200W-26JUL31,If the value of B200 compute per hour is above...
6,KXB200W,B200 Weekly Price,KXB200W-26JUL24-7.24,B200 price up in next week?,finalized,0.0000,1.0000,0.2500,0.2500,637.37,0.00,421.37,0.0000,7.24,2026-07-17T21:00:00Z,2026-07-24T21:00:00Z,2026-07-17T20:30:27.588268Z,2026-07-27T15:35:25.969996Z,KXB200W-26JUL24,If the value of B200 compute per hour is above...
7,KXB200W,B200 Weekly Price,KXB200W-26JUL17-6.69,B200 price up in next week?,finalized,0.0000,1.0000,0.9600,0.9600,1327.66,0.00,710.06,0.0000,6.69,2026-07-10T21:00:00Z,2026-07-17T21:00:00Z,2026-07-10T20:30:25.229059Z,2026-07-17T21:35:32.651407Z,KXB200W-26JUL17,If the value of B200 compute per hour is above...
8,KXB200W,B200 Weekly Price,KXB200W-26JUL10-5.14,B200 price up in next week?,finalized,0.0000,1.0000,0.9900,0.9900,1105.57,0.00,913.00,0.0000,5.14,2026-07-03T21:00:00Z,2026-07-10T21:00:00Z,2026-07-03T20:30:28.825814Z,2026-07-10T21:35:30.330898Z,KXB200W-26JUL10,If the value of B200 compute per hour is above...
9,KXB200W,B200 Weekly Price,KXB200W-26JUL03-4.5,B200 price up in next week?,finalized,0.0000,1.0000,0.9900,0.9900,2709.62,0.00,1485.62,0.0000,4.50,2026-06-26T21:00:00Z,2026-07-03T21:00:00Z,2026-06-26T20:30:24.769624Z,2026-07-03T21:35:24.484298Z,KXB200W-26JUL03,If the value of B200 compute per hour is above...


In [11]:
def get_candlesticks(
    series_ticker,
    market_ticker,
    days=30,
    period_interval=KALSHI_CANDLE_PERIOD_DAILY,
):
    """Daily / hourly / minute OHLC for one market."""
    end_ts = int(time.time())
    start_ts = end_ts - days * 24 * 3600
    path = KALSHI_CANDLESTICKS.format(
        series_ticker=series_ticker,
        ticker=market_ticker,
    )
    data = safe_get(
        path,
        {"start_ts": start_ts, "end_ts": end_ts, "period_interval": period_interval},
    )
    return data.get("candlesticks", []) if data else []


def candles_to_rows(series_ticker, market_ticker, candles):
    rows = []
    for c in candles:
        price = c.get("price") or {}
        yes_bid = c.get("yes_bid") or {}
        yes_ask = c.get("yes_ask") or {}
        rows.append(
            {
                "series_ticker": series_ticker,
                "market_ticker": market_ticker,
                "end_period_ts": c.get("end_period_ts"),
                "end_period_utc": datetime.fromtimestamp(
                    c["end_period_ts"], tz=timezone.utc
                ).isoformat()
                if c.get("end_period_ts")
                else None,
                "volume": c.get("volume_fp"),
                "open_interest": c.get("open_interest_fp"),
                "price_open": price.get("open_dollars"),
                "price_high": price.get("high_dollars"),
                "price_low": price.get("low_dollars"),
                "price_close": price.get("close_dollars"),
                "yes_bid_close": yes_bid.get("close_dollars"),
                "yes_ask_close": yes_ask.get("close_dollars"),
            }
        )
    return rows


In [12]:
# Price history for active markets (start with a sample; set SAMPLE=None for all)
active = markets_df[markets_df["status"] == "active"].copy()
SAMPLE = 20  # None → every active market (slower)

targets = active if SAMPLE is None else active.head(SAMPLE)
history_rows = []
for _, row in targets.iterrows():
    candles = get_candlesticks(
        row["series_ticker"],
        row["market_ticker"],
        days=30,
        period_interval=KALSHI_CANDLE_PERIOD_DAILY,
    )
    history_rows.extend(candles_to_rows(row["series_ticker"], row["market_ticker"], candles))
    print(f"{row['market_ticker']}: {len(candles)} daily candles")

history_df = pd.DataFrame(history_rows)
print(f"\nHistory rows: {len(history_df)}")
history_df.tail(15)


GET /series/KXB200W/markets/KXB200W-26SEP04-6.01/candlesticks -> HTTP 200
KXB200W-26SEP04-6.01: 2 daily candles
GET /series/KXH200MAX/markets/KXH200MAX-26DEC31-6.690/candlesticks -> HTTP 200
KXH200MAX-26DEC31-6.690: 29 daily candles
GET /series/KXH200MAX/markets/KXH200MAX-26DEC31-6.390/candlesticks -> HTTP 200
KXH200MAX-26DEC31-6.390: 29 daily candles
GET /series/KXH200MAX/markets/KXH200MAX-26DEC31-6.090/candlesticks -> HTTP 200
KXH200MAX-26DEC31-6.090: 30 daily candles
GET /series/KXH200MAX/markets/KXH200MAX-26DEC31-5.790/candlesticks -> HTTP 200
KXH200MAX-26DEC31-5.790: 30 daily candles
GET /series/KXNVDAA/markets/KXNVDAA-28JANHEAD-56000/candlesticks -> HTTP 200
KXNVDAA-28JANHEAD-56000: 21 daily candles
GET /series/KXNVDAA/markets/KXNVDAA-28JANHEAD-54000/candlesticks -> HTTP 200
KXNVDAA-28JANHEAD-54000: 27 daily candles
GET /series/KXNVDAA/markets/KXNVDAA-28JANHEAD-52000/candlesticks -> HTTP 200
KXNVDAA-28JANHEAD-52000: 26 daily candles
GET /series/KXNVDAA/markets/KXNVDAA-28JANHEAD-5

,series_ticker,market_ticker,end_period_ts,end_period_utc,volume,open_interest,price_open,price_high,price_low,price_close,yes_bid_close,yes_ask_close
277,KXB200WS,KXB200WS-26OCT09-8.500,1788062400,2026-08-30T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.0200,0.0600
278,KXB200WS,KXB200WS-26OCT09-8.000,1787976000,2026-08-29T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.0600,0.1000
279,KXB200WS,KXB200WS-26OCT09-8.000,1788062400,2026-08-30T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.0500,0.0900
280,KXB200WS,KXB200WS-26OCT09-7.500,1787976000,2026-08-29T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.1000,0.1400
281,KXB200WS,KXB200WS-26OCT09-7.500,1788062400,2026-08-30T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.1000,0.1500
282,KXB200WS,KXB200WS-26OCT09-7.000,1787976000,2026-08-29T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.1900,0.2200
283,KXB200WS,KXB200WS-26OCT09-7.000,1788062400,2026-08-30T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.1900,0.2500
284,KXB200WS,KXB200WS-26OCT09-6.500,1787976000,2026-08-29T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.3300,0.4300
285,KXB200WS,KXB200WS-26OCT09-6.500,1788062400,2026-08-30T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.3200,0.3500
286,KXB200WS,KXB200WS-26OCT09-6.000,1787976000,2026-08-29T04:00:00+00:00,0.00,0.00,NaN,NaN,NaN,NaN,0.5000,0.5100


In [13]:
from config import KALSHI_MARKETS_PRICE_HISTORY_CSV, KALSHI_MARKETS_SNAPSHOT_CSV, ensure_data_dir

ensure_data_dir()

# Persist snapshots for later analysis
markets_df.to_csv(KALSHI_MARKETS_SNAPSHOT_CSV, index=False)
history_df.to_csv(KALSHI_MARKETS_PRICE_HISTORY_CSV, index=False)
print(f"Wrote {len(markets_df)} markets → {KALSHI_MARKETS_SNAPSHOT_CSV}")
print(f"Wrote {len(history_df)} candles → {KALSHI_MARKETS_PRICE_HISTORY_CSV}")

# Peek one market's history
if not history_df.empty:
    sample_ticker = history_df["market_ticker"].iloc[0]
    history_df[history_df["market_ticker"] == sample_ticker]


Wrote 1592 markets → compute_markets_snapshot.csv
Wrote 292 candles → compute_markets_price_history.csv


In [14]:
# Example: full candlestick URL (safe_get takes the path; kalshi_url is for inspection/debug)
series_ticker, market_ticker = "KXB200WS", "KXB200WS-26OCT02-9.000"
kalshi_url(KALSHI_CANDLESTICKS, series_ticker=series_ticker, ticker=market_ticker)

'https://external-api.kalshi.com/trade-api/v2/series/KXB200WS/markets/KXB200WS-26OCT02-9.000/candlesticks'

In [15]:
from config import ORNN_DAILY_INDEX_SNAPSHOT_CSV, ORNN_GPUS, ensure_data_dir
from ornn_client import (
    daily_index_to_rows,
    get_daily_index_all,
    get_forward_curves,
    forward_to_rows,
    test_ornn_connection,
)

# Quick check: key loaded from .env (does not print the key)
import os
assert os.environ.get("ORNN_API_KEY"), "Set ORNN_API_KEY in .env"

if not test_ornn_connection():
    raise RuntimeError(
        "Ornn API unreachable (connect timeout). Try: disable VPN, switch network, "
        "or run in terminal: curl -I https://api.ornnai.com/api/daily-index/all"
    )

# HIGH: save current daily-index snapshot
ensure_data_dir()
snapshot = get_daily_index_all()
snapshot_rows = daily_index_to_rows(snapshot)
pd.DataFrame(snapshot_rows).to_csv(ORNN_DAILY_INDEX_SNAPSHOT_CSV, index=False)
print(f"Saved {len(snapshot_rows)} GPUs → {ORNN_DAILY_INDEX_SNAPSHOT_CSV}")
snapshot


GET https://api.ornnai.com/api/daily-index/all -> HTTP 200
GET https://api.ornnai.com/api/daily-index/all -> HTTP 200
Saved 6 GPUs → ornn_daily_index_snapshot.csv


{'success': True,
 'date': '2026-08-29T20:00:00.000Z',
 'data': [{'gpu_type': 'A100 SXM4',
   'region': 'global',
   'index_value': 1.0316666666666667,
   'date': '2026-08-29T20:00:00.000Z'},
  {'gpu_type': 'B200',
   'region': 'global',
   'index_value': 6.064583333333333,
   'date': '2026-08-29T20:00:00.000Z'},
  {'gpu_type': 'H100 SXM',
   'region': 'global',
   'index_value': 2.872916666666667,
   'date': '2026-08-29T20:00:00.000Z'},
  {'gpu_type': 'H200',
   'region': 'global',
   'index_value': 4.437916666666666,
   'date': '2026-08-29T20:00:00.000Z'},
  {'gpu_type': 'RTX 5090',
   'region': 'global',
   'index_value': 0.5304166666666666,
   'date': '2026-08-29T20:00:00.000Z'},
  {'gpu_type': 'RTX PRO 6000 WS',
   'region': 'global',
   'index_value': 1.2975,
   'date': '2026-08-29T20:00:00.000Z'}],
 'count': 6}

In [18]:
from config import ORNN_GPU_DAILY_HISTORY_CSV, ensure_data_dir
from ornn_client import get_gpu_history_full, records_to_rows, summarize_fields, get_gpu_volatility, get_gpu_volume_metrics

# Full daily history (API default limit=100 was truncating to ~100 rows/GPU)
ornn_history_rows = []
ornn_history_queries = []
for gpu in ORNN_GPUS:
    records, queries = get_gpu_history_full(gpu, granularity="daily")
    ornn_history_queries.extend(queries)
    rows = records_to_rows(gpu, records)
    ornn_history_rows.extend(rows)
    ts_col = "recorded_at" if records and "recorded_at" in records[0] else "timestamp"
    if records:
        print(
            f"{gpu}: {len(rows)} rows  "
            f"({records[0].get(ts_col, '')[:10]} → {records[-1].get(ts_col, '')[:10]})"
        )
    else:
        print(f"{gpu}: 0 rows")

ensure_data_dir()
ornn_history_df = pd.DataFrame(ornn_history_rows)
ornn_history_df.to_csv(ORNN_GPU_DAILY_HISTORY_CSV, index=False)
print(f"\nWrote {len(ornn_history_df)} rows → {ORNN_GPU_DAILY_HISTORY_CSV}")
print("\nFields pulled (history-range):")
print(summarize_fields(ornn_history_rows))
ornn_history_df.groupby("gpu_name").size()



GET https://api.ornnai.com/api/gpu/B200/history-range -> HTTP 404
GET https://api.ornnai.com/api/gpu/B200/history-range -> HTTP 200
GET https://api.ornnai.com/api/gpu/B200/history-range -> HTTP 200
B200: 340 rows  (2025-09-23 → 2026-08-29)
GET https://api.ornnai.com/api/gpu/H100%20SXM/history-range -> HTTP 200
GET https://api.ornnai.com/api/gpu/H100%20SXM/history-range -> HTTP 200
GET https://api.ornnai.com/api/gpu/H100%20SXM/history-range -> HTTP 200
H100 SXM: 796 rows  (2024-06-23 → 2026-08-29)
GET https://api.ornnai.com/api/gpu/H200/history-range -> HTTP 404
GET https://api.ornnai.com/api/gpu/H200/history-range -> HTTP 200
GET https://api.ornnai.com/api/gpu/H200/history-range -> HTTP 200
H200: 583 rows  (2025-01-23 → 2026-08-29)
GET https://api.ornnai.com/api/gpu/A100%20SXM4/history-range -> HTTP 200
GET https://api.ornnai.com/api/gpu/A100%20SXM4/history-range -> HTTP 200
GET https://api.ornnai.com/api/gpu/A100%20SXM4/history-range -> HTTP 200
A100 SXM4: 970 rows  (2024-01-01 → 2026

gpu_name
A100 SXM4          970
B200               340
H100 SXM           796
H200               583
RTX 5090           552
RTX PRO 6000 WS    347
dtype: int64

In [19]:
from config import ORNN_GPU_VOLATILITY_CSV, ORNN_GPU_VOLUME_METRICS_CSV

# MEDIUM: volatility + volume metrics (Premium analytics, all fields preserved)
ornn_vol_rows, ornn_vol_metric_rows = [], []
for gpu in ORNN_GPUS:
    vol = get_gpu_volatility(gpu)
    metrics = get_gpu_volume_metrics(gpu)
    ornn_vol_rows.extend(records_to_rows(gpu, vol))
    ornn_vol_metric_rows.extend(records_to_rows(gpu, metrics))
    print(f"{gpu}: {len(vol)} vol rows, {len(metrics)} volume-metric rows")

ensure_data_dir()
pd.DataFrame(ornn_vol_rows).to_csv(ORNN_GPU_VOLATILITY_CSV, index=False)
pd.DataFrame(ornn_vol_metric_rows).to_csv(ORNN_GPU_VOLUME_METRICS_CSV, index=False)

print("\nVolatility fields:")
print(summarize_fields(ornn_vol_rows))
print("\nVolume-metric fields:")
print(summarize_fields(ornn_vol_metric_rows))



GET https://api.ornnai.com/api/gpu/B200/volatility -> HTTP 404
GET https://api.ornnai.com/api/gpu/B200/volatility -> HTTP 200
GET https://api.ornnai.com/api/gpu/B200/volatility -> HTTP 200
GET https://api.ornnai.com/api/gpu/B200/volume-metrics -> HTTP 200
GET https://api.ornnai.com/api/gpu/B200/volume-metrics -> HTTP 200
GET https://api.ornnai.com/api/gpu/B200/volume-metrics -> HTTP 200
B200: 339 vol rows, 360 volume-metric rows
GET https://api.ornnai.com/api/gpu/H100%20SXM/volatility -> HTTP 200
GET https://api.ornnai.com/api/gpu/H100%20SXM/volatility -> HTTP 200
GET https://api.ornnai.com/api/gpu/H100%20SXM/volatility -> HTTP 200
GET https://api.ornnai.com/api/gpu/H100%20SXM/volume-metrics -> HTTP 200
GET https://api.ornnai.com/api/gpu/H100%20SXM/volume-metrics -> HTTP 200
GET https://api.ornnai.com/api/gpu/H100%20SXM/volume-metrics -> HTTP 200
H100 SXM: 795 vol rows, 811 volume-metric rows
GET https://api.ornnai.com/api/gpu/H200/volatility -> HTTP 404
GET https://api.ornnai.com/api/

In [20]:
from config import ORNN_FORWARD_CURVES_CSV
from ornn_client import get_forward_curves, forward_to_rows, summarize_fields

# MEDIUM: forward curves (future compute pricing by tenor)
forward = get_forward_curves()
forward_rows = forward_to_rows(forward)
forward_df = pd.DataFrame(forward_rows)
ensure_data_dir()
forward_df.to_csv(ORNN_FORWARD_CURVES_CSV, index=False)
print(f"Wrote {len(forward_df)} rows → {ORNN_FORWARD_CURVES_CSV}")
print(summarize_fields(forward_rows))
forward_df


GET https://api.ornnai.com/api/forward -> HTTP 200
Wrote 40 rows → ornn_forward_curves.csv
{'as_of_date': 'str', 'curve_as_of': 'str', 'gpu_name': 'str', 'has_mark': 'bool', 'insufficient_data': 'NoneType', 'label': 'str', 'months': 'int', 'price': 'float', 'provenance_sample_count': 'NoneType', 'provenance_source': 'str', 'provenance_updated_at': 'str'}


,gpu_name,label,months,price,has_mark,as_of_date,provenance_source,provenance_updated_at,provenance_sample_count,insufficient_data,curve_as_of
0,H100 SXM,1M,1,2.61,True,2026-08-13T23:39:08.128Z,hand_entered_mark,2026-08-13T23:39:08.128Z,None,None,2026-08-25T02:39:04.063Z
1,H100 SXM,6M,6,2.48,True,2026-08-13T23:39:08.128Z,hand_entered_mark,2026-08-13T23:39:08.128Z,None,None,2026-08-25T02:39:04.063Z
2,H100 SXM,1Y,12,2.25,True,2026-08-13T23:39:08.128Z,hand_entered_mark,2026-08-13T23:39:08.128Z,None,None,2026-08-25T02:39:04.063Z
3,H100 SXM,3Y,36,1.78,True,2026-08-13T23:39:08.128Z,hand_entered_mark,2026-08-13T23:39:08.128Z,None,None,2026-08-25T02:39:04.063Z
4,H100 SXM,5Y,60,1.56,True,2026-08-13T23:39:08.128Z,hand_entered_mark,2026-08-13T23:39:08.128Z,None,None,2026-08-25T02:39:04.063Z
5,H200,1M,1,4.49,True,2026-08-13T23:39:19.307Z,hand_entered_mark,2026-08-13T23:39:19.307Z,None,None,2026-08-25T02:39:04.063Z
6,H200,6M,6,4.15,True,2026-08-13T23:39:19.307Z,hand_entered_mark,2026-08-13T23:39:19.307Z,None,None,2026-08-25T02:39:04.063Z
7,H200,1Y,12,3.60,True,2026-08-13T23:39:19.307Z,hand_entered_mark,2026-08-13T23:39:19.307Z,None,None,2026-08-25T02:39:04.063Z
8,H200,3Y,36,2.29,True,2026-08-13T23:39:19.307Z,hand_entered_mark,2026-08-13T23:39:19.307Z,None,None,2026-08-25T02:39:04.063Z
9,H200,5Y,60,1.96,True,2026-08-13T23:39:19.307Z,hand_entered_mark,2026-08-13T23:39:19.307Z,None,None,2026-08-25T02:39:04.063Z
